# Verifikasi hasil crawl penuh

Jalankan sel di bawah, lalu **copy-paste seluruh output teksnya** ke chat.
Tidak ada data mentah/HTML yang perlu dikirim — hanya ringkasan angka.

In [ ]:
import os
from pathlib import Path
import pandas as pd

if Path('src').exists():
    ROOT = Path('.').resolve()
elif Path('../src').exists():
    os.chdir('..'); ROOT = Path('.').resolve()
INTERIM = ROOT / 'data/interim'
RAW = ROOT / 'data/raw'

links = pd.read_parquet(INTERIM / 'news_links.parquet')
clean = pd.read_parquet(INTERIM / 'news_clean_raw.parquet')

print('===== RINGKASAN CRAWL =====')
print('total link            :', len(links))
print('total kandidat        :', int(links.is_candidate.sum()))
print('total html_saved      :', int(links.html_saved.sum()))
print('artikel ter-parse     :', len(clean))
print('file .gz di disk      :', len(list((RAW/"html_article").rglob("*.html.gz"))))

print('\n===== RENTANG TANGGAL (cek sampai 2026-09-30?) =====')
print('index_date min        :', links.index_date.min())
print('index_date max        :', links.index_date.max())
print('jumlah hari unik      :', links.index_date.nunique())

print('\n===== DISTRIBUSI PER BULAN (deteksi lubang) =====')
links['ym'] = links.index_date.str[:7]
per_month = links.groupby('ym').agg(link=('article_id','size'), kandidat=('is_candidate','sum'))
print(per_month.to_string())

print('\n===== GAGAL PARSE / ANOMALI =====')
print('kandidat tanpa html_saved :', int((links.is_candidate & ~links.html_saved).sum()))
print('published_at kosong       :', int(clean.published_at.isna().sum()))
print('n_words < 40              :', int((clean.n_words < 40).sum()))
print('n_words statistik         : min', clean.n_words.min(), '| median', int(clean.n_words.median()), '| max', clean.n_words.max())

print('\n===== DISTRIBUSI SUBCHANNEL (top 15) =====')
print(links.subchannel.value_counts().head(15).to_string())

In [ ]:
# Perkiraan sebaran bank (hitungan kasar via alias di judul; final-nya di Fase 4)
import re, yaml
al = yaml.safe_load(open(ROOT / 'config/aliases.yaml'))
print('===== PERKIRAAN SEBUTAN BANK DI JUDUL (kasar) =====')
for bank, pats in al['banks'].items():
    rx = re.compile('|'.join(pats), re.IGNORECASE)
    n = links.title.fillna('').str.contains(rx).sum()
    print(f'{bank}: {int(n)} judul')